MOMENTO 1:COMPRENSION DEL PROBLEMA Y RECONOCIMIENTO DE LOS DATOS

3. Identifique los cinco archivos suministrados y explique qué información considera que
almacena cada uno.

instacart_orders.csv:Los pedidos hechos en la app

productos.csv:El catalogo de productos

order_products.csv:El detalle de cada pedido

aisles.csv:Los nombres de los pasillos.

departments.csv:Los nombres de los departamentos



4. Reconozca qué columnas podrían utilizarse para identificar pedidos, usuarios y productos.

Para identificar pedidos: order_id. Aparece en instacart_orders (una fila por pedido) y en order_products (una fila por articulo del pedido).

Para identificar usuarios: user_id. Aparece en instacart_orders y permite identificar la cuenta de cada cliente.

Para identificar productos: product_id. Aparece en products (una fila por producto) y en order_products

5. Formule tres preguntas de análisis relacionadas con los hábitos de compra que puedan
responderse mediante las operaciones básicas estudiadas.

1. ¿En que dia de la semana y a que hora se hacen mas pedidos?

2. ¿Cuanto tiempo suelen esperar los clientes entre un pedido y el siguiente?

3. ¿Cuales son los productos que mas se piden y cuales se recompran con mas frecuencia?


Registre en su Notebook la descripción del problema, las preguntas y una explicación sobre la
importancia de explorar los datos antes de analizarlos.

- Descripcion del problema :

Instacart es una plataforma de entrega de viveres a domicilio y  a mi como analista de datos me piden explorar los pedidos para entender como compran los clientes.

- Importancia de explorar los datos antes de analizarlos:

Antes de empezar a analizar datos es de suma importancia conocer los datos y el problema para conocer el objetivo de el analisis.



MOMENTO 2 :IMPORTACION Y EXPLORACION INICIAL DE LOS ARCHIVOS

In [2]:
import sys
import pandas as pd

print(sys.version_info.major, sys.version_info.minor)
print(pd.__version__)

3 13
3.1.0


In [3]:
import pandas as pd

orders = pd.read_csv("../data/instacart_orders.csv", sep=";")
products = pd.read_csv("../data/products.csv", sep=";")
aisles = pd.read_csv("../data/aisles.csv", sep=";")
departments = pd.read_csv("../data/departments.csv", sep=";")
order_products = pd.read_csv("../data/order_products.csv", sep=";")

6. Utilice head() y tail() para observar los primeros y últimos cinco registros de cada DataFrame.

In [4]:
print("orders - primeros 5")
display(orders.head())
print("orders - últimos 5")
display(orders.tail())

orders - primeros 5


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


orders - últimos 5


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


7. Utilice shape para identificar la cantidad de filas y columnas.

In [7]:
dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

In [8]:
for nombre, df in dataframes.items():
    print(f"{nombre}: {df.shape[0]} filas y {df.shape[1]} columnas")

orders: 478967 filas y 6 columnas
products: 49694 filas y 4 columnas
aisles: 134 filas y 2 columnas
departments: 21 filas y 2 columnas
order_products: 4545007 filas y 4 columnas


¿Cuál de los cinco DataFrames contiene más registros?

El DataFrame con mayor cantidad de registros es  order_products, con 4,545,007 filas. 

¿Cuál contiene menos columnas?

Los DataFrames con menos columnas son aisles y departments, con 2 columnas cada uno. Esto se debe a que son tablas de consulta: solo guardan un identificador (aisle_id o department_id) y el nombre correspondiente (aisle o department). 

¿Qué diferencias identifica entre los datos de pedidos y los datos de productos?


La informacion de los pedidos(orders, con 478,967 filas y 6 columnas) describen el comportamiento de compra: quién hizo cada pedido, en qué día y hora, y cuánto tiempo pasó desde el anterior. Los datos de productos (products, con 49,694 filas y 4 columnas) describen el catálogo: el nombre de cada producto y el pasillo y departamento al que pertenece. Los pedidos son eventos que se repiten en el tiempo, mientras que los productos son un catálogo de referencia. Ambos se conectan a través de order_products, que indica qué productos hay en cada pedido.

¿Por qué es importante conocer las dimensiones de un conjunto de datos?

Las razones principales:

- Verificar la importacion : si una tabla sale con una sola columna ,fallo el ";" .

Si sale con muy pocas filas,el archivo pudo leerse incompleto

- Dimensionar el trabajo: No es lo mismo procesar 21 filas (departments) que 4,545,007 (order_products).

Con tablas grandes se gasta mas memoria.

- Planear el analisis : saber cuántas columnas hay indica qué variables puedes usar. Por ejemplo, orders tiene 6 columnas, suficientes para analizar día, hora y frecuencia de compra.

- Entender como se relacionan las tablas:las proporciones dan pistas. order_products tiene casi diez veces más filas que orders, lo que indica que cada pedido tiene en promedio varios artículos.

- Comprobar resultados despues : si al unir tablas con merge() el numero de filas cambia de forma inesperada,es señal de duplicados o de una unión mal hecha.

MOMENTO 3 : RECONOCIMIENTO DE LA ESTRUCTURA Y CALIDAD INICIAL DE LOS DATOS

- Identifique los nombres de las columnas de cada archivo.

In [11]:
for nombre, df in dataframes.items():
    print(f"{nombre}: {list(df.columns)}")

orders: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
products: ['product_id', 'product_name', 'aisle_id', 'department_id']
aisles: ['aisle_id', 'aisle']
departments: ['department_id', 'department']
order_products: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


- Determine qué columnas contienen valores enteros, decimales o texto.

In [12]:
for nombre, df in dataframes.items():
    print(f"===== {nombre} =====")
    print("Enteros:  ", list(df.select_dtypes(include="int").columns))
    print("Decimales:", list(df.select_dtypes(include="float").columns))
    print("Texto:    ", list(df.select_dtypes(include="object").columns))
    print()

===== orders =====
Enteros:   ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day']
Decimales: ['days_since_prior_order']
Texto:     []

===== products =====
Enteros:   ['product_id', 'aisle_id', 'department_id']
Decimales: []
Texto:     ['product_name']

===== aisles =====
Enteros:   ['aisle_id']
Decimales: []
Texto:     ['aisle']

===== departments =====
Enteros:   ['department_id']
Decimales: []
Texto:     ['department']

===== order_products =====
Enteros:   ['order_id', 'product_id', 'reordered']
Decimales: ['add_to_cart_order']
Texto:     []



C:\Users\Juan David Salazar T\AppData\Local\Temp\ipykernel_17000\3684118483.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print("Texto:    ", list(df.select_dtypes(include="object").columns))
C:\Users\Juan David Salazar T\AppData\Local\Temp\ipykernel_17000\3684118483.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See htt

- Identifique cuáles son los campos que permiten relacionar conceptualmente los diferentes
conjuntos de datos.

In [13]:
from itertools import combinations

for (nombre1, df1), (nombre2, df2) in combinations(dataframes.items(), 2):
    comunes = set(df1.columns) & set(df2.columns)
    if comunes:
        print(f"{nombre1} ↔ {nombre2}: {sorted(comunes)}")

orders ↔ order_products: ['order_id']
products ↔ aisles: ['aisle_id']
products ↔ departments: ['department_id']
products ↔ order_products: ['product_id']


- Examine el número de valores no nulos reportado por info().

In [14]:
for nombre, df in dataframes.items():
    print(f"===== {nombre} =====")
    df.info()
    print()

===== orders =====
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

===== products =====
<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)
mem

- Compare el número de registros con la cantidad de valores no nulos para reconocer posibles
datos ausentes.

In [15]:
for nombre, df in dataframes.items():
    print(f"===== {nombre} =====")
    comparacion = pd.DataFrame({
        "registros": len(df),
        "no_nulos": df.count(),
    })
    comparacion["ausentes"] = comparacion["registros"] - comparacion["no_nulos"]
    comparacion["% ausentes"] = (comparacion["ausentes"] / comparacion["registros"] * 100).round(2)
    display(comparacion)

===== orders =====


,registros,no_nulos,ausentes,% ausentes
order_id,478967,478967,0,0.00
user_id,478967,478967,0,0.00
order_number,478967,478967,0,0.00
order_dow,478967,478967,0,0.00
order_hour_of_day,478967,478967,0,0.00
days_since_prior_order,478967,450148,28819,6.02


===== products =====


,registros,no_nulos,ausentes,% ausentes
product_id,49694,49694,0,0.00
product_name,49694,48436,1258,2.53
aisle_id,49694,49694,0,0.00
department_id,49694,49694,0,0.00


===== aisles =====


,registros,no_nulos,ausentes,% ausentes
aisle_id,134,134,0,0.0
aisle,134,134,0,0.0


===== departments =====


,registros,no_nulos,ausentes,% ausentes
department_id,21,21,0,0.0
department,21,21,0,0.0


===== order_products =====


,registros,no_nulos,ausentes,% ausentes
order_id,4545007,4545007,0,0.00
product_id,4545007,4545007,0,0.00
add_to_cart_order,4545007,4544171,836,0.02
reordered,4545007,4545007,0,0.00


- Elabore una tabla de diagnóstico con el nombre del archivo, cantidad de filas, cantidad de
columnas y observaciones sobre los datos.

In [16]:
filas_tabla = []

for nombre, df in dataframes.items():
    ausentes = df.isna().sum()
    ausentes = ausentes[ausentes > 0]
    duplicados = df.duplicated().sum()

    observaciones = []
    if ausentes.empty:
        observaciones.append("Sin datos ausentes")
    else:
        detalle = ", ".join(f"{col} ({n})" for col, n in ausentes.items())
        observaciones.append(f"Datos ausentes en: {detalle}")
    observaciones.append(f"{duplicados} filas duplicadas")

    filas_tabla.append({
        "Archivo": f"{nombre}.csv" if nombre != "orders" else "instacart_orders.csv",
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Observaciones": "; ".join(observaciones),
    })

diagnostico = pd.DataFrame(filas_tabla)
diagnostico

,Archivo,Filas,Columnas,Observaciones
0,instacart_orders.csv,478967,6,Datos ausentes en: days_since_prior_order (288...
1,products.csv,49694,4,Datos ausentes en: product_name (1258); 0 fila...
2,aisles.csv,134,2,Sin datos ausentes; 0 filas duplicadas
3,departments.csv,21,2,Sin datos ausentes; 0 filas duplicadas
4,order_products.csv,4545007,4,Datos ausentes en: add_to_cart_order (836); 0 ...


PREGUNTAS DE ANALISIS:

¿Qué diferencias existen entre int64, float64 y object?


int64 almacena números enteros, float64 números con decimales (y también columnas numéricas con valores ausentes) y object almacena texto.

¿Qué información proporciona info() que no se obtiene directamente con shape?

shape solo indica el número de filas y columnas. info() además muestra el nombre y el tipo de dato de cada columna y la cantidad de valores no nulos, lo que permite detectar datos ausentes.

¿En qué columnas se observan valores ausentes?

- days_since_prior_order (en orders): 28,819 valores ausentes de 478,967 registros (aprox. 6.02 %).
- product_name (en products): 1,258 valores ausentes de 49,694 registros (aprox. 2.53 %).
- add_to_cart_order (en order_products): 836 valores ausentes de 4,545,007 registros (aprox. 0.02 %).
Los DataFrames aisles y departments no presentan datos ausentes.

¿Qué posibles dificultades podrían generar estos valores en un análisis posterior?


Los valores ausentes pueden sesgar promedios y estadísticas, por ejemplo el promedio de días entre pedidos, que solo se calcularía con los pedidos que sí tienen dato. También pueden producir conteos incompletos, como productos sin nombre en un ranking de productos más pedidos, y cambiar el tipo de dato de una columna. Si no se revisan antes de analizar, pueden llevar a conclusiones erróneas.

MOMENTO 4: INDEXACION Y FILTRADO LOGICO

1. EJERCICIO

In [18]:
ej1 = products[["product_id", "product_name"]]
ej1.head()

,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


Sirve para ver solo dos columnas de productos

2. EJERCICIO

In [19]:
ej2 = products[products["department_id"] == 1]
print(len(ej2))
ej2.head()

4008


,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


Se ven productos del departamento 1

3. EJERCICIO

In [20]:
ej3 = orders[orders["order_hour_of_day"] == 10]
print(len(ej3))
ej3.head()

40578


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


Pedidos hechos a las 10 de la mañana

4. EJERCICIO

In [21]:
ej4 = orders[orders["order_dow"] == 0]
print(len(ej4))
ej4.head()

84090


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


Pedidos hecho el domingo

5. EJERCICIO 

In [22]:
ej5 = orders[orders["order_number"] > 10]
print(len(ej5))
ej5.head()

244072


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


Pedidos de cliente que ya llevan mas de 10 pedidos.

6. EJERCICIO

In [23]:
ej6 = orders[orders["days_since_prior_order"] > 15]
print(len(ej6))
ej6.head()

110706


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


Pedidos que se hicieron hace 15 dias despues del anterior.

7. EJERCICIO

In [24]:
ej7 = order_products[order_products["reordered"] == 1]
print(len(ej7))
ej7.head()

2683838


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


Productos que el cliente ya habia comprado antes

8.EJERCICIO

In [25]:
ej8 = order_products[order_products["add_to_cart_order"] == 1]
print(len(ej8))
ej8.head()

450046


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


Porductos que fueron lo primero que se pusieron en el camino

POSTERIOR MENTE APLIQUE DOS FILTROS CON CONDICIONES SIMULTANEAS

Pedidos realizados a la 10 horas y con order_number superior a 10

In [4]:
filtro1 = orders[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
print(len(filtro1))
filtro1.head()

21537


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


Productos que pertenezcan al departamente 1 y que tengan un aisle_id superior a 50

In [5]:
filtro2 = products[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
print(len(filtro2))
filtro2.head()

1618


,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


PREGUNTAS DE ANALISIS

¿Qué diferencia existe entre seleccionar columnas y filtrar filas?

Seleccionar columnas disminuye el DataFrame en hroizontal:eliges que variables deseas ver y conservas todas las filas.Filtrar filas lo reduce en vertical:eliges que filtros quieres segun una condicion y conservas todas las columnas

¿Qué ventajas ofrece loc[] al consultar información específica?

loc[] permite filtrar filas y seleccionar columnas en una sola instruccion 


Ventajas principales:

- Consulta precisa en un paso:combina condiciones y columnas sin encadenar las operaciones.

- Codigo mas legible:Se ve de inmediato que colunmas y filas se piden.

- Seleccion por etiquetas:usas los nombres verdaderos de la columnas y del indice ,no posiciones numericas.

- Rangos incluyentes:en loc[0:4] se incluye el 4,a diferencia de iloc , que lo excluye

- Asignacion segura:al modificar valores, loc evita advertencias.







¿Que ocurre con la cantidad de registros cuando se aplican varias condiciones?

La cantidad de registros se diminuye o queda igual ,nunca aumenta.

¿Por qué es importante utilizar correctamente los operadores lógicos?

Porque un operador erroneo cambia que registros se seleccionan y produce resultados erroneos,muchas veces sin mostrar ningun error.

MOMENTO 5:OBTENCION E INTERPRETACION DE INDICADORES BASICOS 

¿Cuántos registros de pedidos contiene el archivo?

In [6]:
print(orders.shape[0])
print(orders["order_id"].count())

478967
478967


¿Cuántos registros de productos contiene el catálogo?

In [7]:
print(products.shape[0])

49694


¿Cuál es el promedio de order_number?

In [8]:
print(orders["order_number"].mean())

17.157808366755955


¿Cuál es el promedio de days_since_prior_order entre los valores disponibles?

In [9]:
print(orders["days_since_prior_order"].mean())

11.101750979677794


¿Cuántos registros de artículos tienen reordered igual a 1?

In [10]:
recompras = order_products[order_products["reordered"] == 1]
print(len(recompras))

2683838


¿Cuál es la suma de la columna reordered?

In [11]:
print(order_products["reordered"].sum())

2683838


¿Qué características estadísticas presenta order_hour_of_day?

In [12]:
print(orders["order_hour_of_day"].describe())

count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64


¿Qué características estadísticas presenta add_to_cart_order?

In [13]:
print(order_products["add_to_cart_order"].describe())

count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64


TABLAS

TABLA DE INDICADORES

![alt text](../imagenes/tabla_de_indicadores.png)

DIFERENCIA ENTRE PEDIDOS REGISTROS Y USUARIOS

![alt text](../imagenes/registros_pedidos_usuarios.png)

MOMENTO 6: RESOLUCION DEL PROBLEMA Y REPRESENTACION DEL PRODUCTO

SINTESIS INDIVIDUAL

1 PREGUNTAS DE ANALISIS Y RESPUESTAS

¿En que dia de la semana y a que horas se realizan mas pedidos?

El dia con mas pedidos es el domingo ,(84,090),seguido del lunes (82,185),el jueves es el dia con menos (59,810).Por hora ,los pedidos se concentran entre las 10 y las 15vhoras:las horas con mas pedidos son la 10 (40,578),la 11 (40,032) y la 15 (39,789).

La mitad de los pedidos se hace entre las 10 y las 16 horas con mediana en la hora 13.

¿Cuantos diaS pasan normalmente entre un pedido y el siguiente?

En promedio pasan 11.10 dias entre pedido y el siguiente,El calculo se hizo con 450,148 valores disponibles,porque 28,819 pedidos no tienen dato en esta columna.

¿Cuales son los productos mas pedidos y los que mas se recompran?

Los productos mas pedidos son Banana (66,050),Bag Of Organic Bananas (53,297) y organic Strawberries (37,039).Entre los diez mas pedidos,lso de mayor tasa de recompra son Banana (0.84),Hag Of Organic Bananas (0.83) y Organic Whole Milk(0.83) y le menor tasa es limes (0.68). En general 2,683,838 de los 4,545,007 articulos registrados son recompras ,cerca del 59%.

2. Principales hallazgos

Pedidos

•	orders tiene 478,967 registros, pero solo 478,952 pedidos distintos y 157,437 usuarios distintos. El número de filas no equivale al número de clientes.

•	Cada cliente aparece en promedio en unos 3 pedidos del archivo, aunque el promedio de order_number es 17.16. Esto indica que el archivo contiene solo una parte del historial de cada cliente.

•	Hay 244,072 pedidos con order_number superior a 10, es decir, cerca de la mitad de los registros.

•	Los pedidos se concentran en la franja de las 10 a las 16 horas y son más frecuentes el domingo y el lunes.


Productos

•	El catálogo tiene 49,694 productos y order_products tiene 4,545,007 registros, uno por cada artículo dentro de un pedido.


•	Los productos más pedidos son frutas, verduras y lácteos, y todos tienen una tasa de recompra entre 0.68 y 0.84.


•	La mitad de los artículos se agrega en las primeras 6 posiciones del carrito, y los pedidos llegan hasta 64 artículos.

3. COMO AYUDARON LA INDEXACION Y LOS FILTROS

Seleccionar columnas permitió ver solo la información necesaria, por ejemplo product_id y product_name. Los filtros permitieron consultar grupos específicos: 40,578 pedidos hechos a las 10 horas, 244,072 pedidos con order_number superior a 10 y 4,008 productos del departamento 1.
Al combinar condiciones con el operador &, el resultado se redujo: 21,537 pedidos cumplen ambas condiciones (hora 10 y order_number mayor a 10), frente a 40,578 y 244,072 por separado. Lo mismo ocurrió con los productos: de los 4,008 del departamento 1, solo 1,618 tienen además un aisle_id superior a 50.
Por último, loc[] permitió filtrar filas y elegir columnas en una sola instrucción, lo que hizo las consultas más precisas.


4. lIMITACIONES IDENTIFICADAS


•	Filas repetidas: 15 order_id aparecen dos veces en orders, con valores idénticos en todas las columnas. Por eso las filas (478,967) no coinciden con los pedidos distintos (478,952).

•	Valores ausentes: days_since_prior_order tiene 28,819, product_name tiene 1,258 y add_to_cart_order tiene 836. Los promedios solo representan a los registros con dato, y los productos sin nombre no aparecen en el ranking de más pedidos.

•	Pedidos sin detalle: orders tiene 478,952 pedidos distintos, pero solo 450,046 aparecen en order_products; faltan 28,906 pedidos en el detalle.

•	Historial incompleto: el promedio de order_number (17.16) es mucho mayor que el número de pedidos por cliente en el archivo (cerca de 3), por lo que no se puede conocer la frecuencia total de compra de cada cliente.

5. CONCLUSIONES

1.	La demanda se concentra en ciertos momentos. Los pedidos son más frecuentes el domingo (84,090) y el lunes (82,185) y se concentran entre las 10 y las 16 horas, con la hora 10 como la de mayor número de pedidos (40,578).



2.	Los clientes son recurrentes y recompran con frecuencia. Pasan en promedio 11.10 días entre pedidos, el 59 % de los artículos pedidos son recompras y los productos más pedidos (bananos, fresas, aguacates, espinacas y leche) tienen tasas de recompra entre 0.68 y 0.84.

3.	La calidad de los datos debe revisarse antes de analizar. Los registros repetidos, los valores ausentes y la diferencia entre filas, pedidos y usuarios muestran que interpretar el número de filas como número de clientes llevaría a conclusiones erróneas.






